In [6]:
##!pip install anthropic

In [7]:
##!pip install python-dotenv

In [1]:
import os
import json
import re
from pathlib import Path
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv() 

TEXT_DIR = Path("data/processed/raw_text")
JSON_DIR = Path("data/processed/extracted_json")
JSON_DIR.mkdir(parents=True, exist_ok=True)

# Set your API key as an environment variable before running:
# export ANTHROPIC_API_KEY="your-key-here"
client = Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

In [2]:
EXTRACTION_PROMPT = """You are an invoice data extraction system for German businesses.

Extract the following fields from the invoice text below. Return ONLY valid JSON,
no explanation, no markdown formatting, no code fences.

Fields to extract:
- vendor: company/sender name
- invoice_number: the invoice/Rechnungsnummer
- invoice_date: in format YYYY-MM-DD
- net_amount: number, no currency symbol
- vat_rate: number (e.g. 19 or 7)
- vat_amount: number, no currency symbol
- total_amount: number, no currency symbol
- currency: 3-letter code (e.g. EUR)

If a field cannot be found, set its value to null.

Invoice text:
---
{invoice_text}
---

Return only the JSON object."""

In [5]:
def extract_fields_from_text(invoice_text):
    message = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=500,
        messages=[
            {"role": "user", "content": EXTRACTION_PROMPT.format(invoice_text=invoice_text)}
        ]
    )

    raw_response = message.content[0].text.strip()

    # Safety: strip markdown code fences if the model adds them anyway
    raw_response = re.sub(r"^```json\s*|\s*```$", "", raw_response).strip()

    try:
        data = json.loads(raw_response)
        return data
    except json.JSONDecodeError:
        print("Failed to parse JSON. Raw response was:")
        print(raw_response)
        return None
        

In [7]:
def process_all_text_files():
    results = {}
    text_files = list(TEXT_DIR.glob("*.txt"))

    if not text_files:
        print("No text files found in", TEXT_DIR)
        return results

    for text_file in text_files:
        invoice_text = text_file.read_text(encoding="utf-8")
        print(f"Extracting fields from: {text_file.name}")

        fields = extract_fields_from_text(invoice_text)

        if fields:
            results[text_file.stem] = fields

            # Save individual JSON file
            out_path = JSON_DIR / f"{text_file.stem}.json"
            out_path.write_text(json.dumps(fields, indent=2, ensure_ascii=False), encoding="utf-8")

            print(json.dumps(fields, indent=2, ensure_ascii=False))
            print()

    return results


extracted_fields = process_all_text_files()


Extracting fields from: invoice_002.txt
{
  "vendor": "Software Solutions AG",
  "invoice_number": "INV-9981",
  "invoice_date": "2026-07-10",
  "net_amount": 1200.0,
  "vat_rate": 19,
  "vat_amount": 228.0,
  "total_amount": 1428.0,
  "currency": "EUR"
}

Extracting fields from: invoice_003.txt
{
  "vendor": "Bio-Catering Berlin",
  "invoice_number": "BC-2026-077",
  "invoice_date": "2026-07-12",
  "net_amount": 85.5,
  "vat_rate": 7,
  "vat_amount": 5.99,
  "total_amount": 91.49,
  "currency": "EUR"
}

Extracting fields from: invoice_001.txt
{
  "vendor": "Buerobedarf Mueller GmbH",
  "invoice_number": "RE-2026-0451",
  "invoice_date": "2026-07-05",
  "net_amount": 250.0,
  "vat_rate": 19,
  "vat_amount": 47.5,
  "total_amount": 297.5,
  "currency": "EUR"
}



In [8]:
print(f"Successfully extracted fields from {len(extracted_fields)} invoice(s)")
print(f"JSON files saved to: {JSON_DIR.resolve()}")


Successfully extracted fields from 3 invoice(s)
JSON files saved to: /Users/akashkumarsamantray/invoice-data-extractor-ai/data/processed/extracted_json


In [ ]:
## Step 2: Structured Field Extraction with Claude API (`02_extract_fields.ipynb`)

### What this notebook does
This is the second stage of the pipeline. It takes the raw invoice text produced in Step 1 and uses the Claude API to extract structured data — vendor, invoice number, date, VAT rate, amounts — as clean JSON.

### Process
1. **Load environment variables** — the Anthropic API key is loaded securely from a local `.env` file (never hardcoded, never committed to git) using `python-dotenv`.
2. **Define an extraction prompt** — a strict instruction prompt tells Claude exactly which fields to extract and to return *only* valid JSON, with `null` for any field it can't find. This avoids the model adding explanations, markdown formatting, or guessing at missing data.
3. **Call the Claude API per invoice** — each `.txt` file from Step 1 is sent to Claude (`claude-sonnet-4-5`), and the model's response is parsed as JSON. A safety step strips accidental markdown code fences in case the model adds them despite instructions.
4. **Save structured output** — each invoice's extracted fields are saved as an individual `.json` file in `data/processed/extracted_json/`, ready for the next step.

### Why an LLM instead of regex/rule-based extraction
Invoice layouts vary a lot between vendors — field labels, ordering, and formatting are inconsistent. A rule-based parser would need custom logic per invoice template. An LLM can generalize across formats from a single well-defined prompt, which is far more scalable for a real business with many different suppliers.

### Design choices worth noting
- **JSON-only output** enforced via prompt instructions makes the output machine-readable and directly usable downstream, without extra text-cleaning logic.
- **API key handled via `.env`** — a basic but important security practice, especially relevant for a project meant to show production-readiness, not just a notebook demo.
- **One file per invoice** keeps the pipeline modular and debuggable — if one invoice fails extraction, it doesn't affect the others.

### Output
- `data/processed/extracted_json/` — one structured JSON file per invoice, e.g.:
```json
{
  "vendor": "Buerobedarf Mueller GmbH",
  "invoice_number": "RE-2026-0451",
  "invoice_date": "2026-07-05",
  "net_amount": 250.00,
  "vat_rate": 19,
  "vat_amount": 47.50,
  "total_amount": 297.50,
  "currency": "EUR"
}
```

### Next step
`03_validate.ipynb` — check the extracted data for consistency (correct VAT math, valid dates, valid VAT rates) before it gets stored, to catch any extraction errors from the LLM.